# NB23 Game theory — Chapter 23

A chapter notebook for *Blockchain for Lawyers: A Reading Through Technology and Economics*. The code is the same as in the text.

- All code here is a **toy implementation** meant to make a principle visible. Do not use it with real assets, real private keys or real transactions.
- No external API is called, and everything runs without a network. Cells that need the bundled snapshot data carry that data inside the cell.
- Standard library only (run and checked on Python 3.13). There is no install cell.
- The outputs are the editor's own run, stored as produced. Running the cells again should give the same values (random seeds are fixed).

In [ ]:
# NB23-1: selfish-mining Monte Carlo — run the Eyal–Sirer state machine (lead 0, 0′, 1, 2, …) event by event and measure the pool's relative revenue R_pool (for study)
import random

def selfish_mining_share(alpha: float, gamma: float, n_events: int, seed: int = 23) -> float:
    """Draw n_events block-discovery events (the pool finds the block with probability α). Returns: the pool's fraction of the blocks left in the main chain."""
    rng = random.Random(seed)
    lead = 0                 # the pool's private lead (in blocks)
    tie = False              # state 0′: two public branches of length 1 are competing
    pool_blocks = others_blocks = 0
    for _ in range(n_events):
        pool_found = rng.random() < alpha
        if tie:                                   # state 0′ — all three paths lead to state 0
            if pool_found:                        # (b) the pool finds a block on its own branch → pool 2
                pool_blocks += 2
            elif rng.random() < gamma:            # (c) the others find a block on the pool's branch → pool 1, others 1
                pool_blocks += 1; others_blocks += 1
            else:                                 # (d) the others find a block on their own branch → others 2
                others_blocks += 2
            tie = False; lead = 0
            continue
        if pool_found:                            # (a) the pool finds a block → lead +1 (reward comes later)
            lead += 1
            continue
        # the others find a block
        if lead == 0:                             # (e) no private branch → others 1
            others_blocks += 1
        elif lead == 1:                           # (f) lead 1 → the pool publishes its hidden block, a race begins
            tie = True; lead = 0
        elif lead == 2:                           # (g) lead 2 → the pool publishes two blocks, the others' block is discarded → pool 2
            pool_blocks += 2; lead = 0
        else:                                     # (h) lead > 2 → the pool publishes one block (pool 1), the others' block is eventually discarded
            pool_blocks += 1; lead -= 1
    return pool_blocks / (pool_blocks + others_blocks)

def R_pool_closed_form(a: float, g: float) -> float:
    """CACM version, equation (3) = arXiv v5, equation (8): R_pool = [α(1−α)²(4α+γ(1−2α)) − α³] / [1 − α(1+(2−α)α)], 0 ≤ α ≤ 1/2"""
    return (a * (1 - a) ** 2 * (4 * a + g * (1 - 2 * a)) - a ** 3) / (1 - a * (1 + (2 - a) * a))

N = 400_000
print(f"{N:,} events. Under honest mining the relative revenue is α itself. Below: the relative revenue under selfish mining (simulation / closed form).")
print(f"{'α':>5} {'γ':>5} {'sim R_pool':>12} {'formula R_pool':>14} {'honest α':>8} {'selfish better?':>15}")
for g in [0.0, 0.5, 1.0]:
    for a in [0.10, 0.20, 0.25, 0.30, 0.34, 0.40, 0.45]:
        sim = selfish_mining_share(a, g, N)
        cf = R_pool_closed_form(a, g)
        print(f"{a:>5.2f} {g:>5.1f} {sim:>12.4f} {cf:>14.4f} {a:>8.2f} {'yes' if cf > a else 'no':>15}")

400,000 events. Under honest mining the relative revenue is α itself. Below: the relative revenue under selfish mining (simulation / closed form).
    α     γ   sim R_pool formula R_pool honest α selfish better?
 0.10   0.0       0.0358         0.0356     0.10              no
 0.20   0.0       0.1300         0.1297     0.20              no
 0.25   0.0       0.1948         0.1951     0.25              no
 0.30   0.0       0.2713         0.2731     0.30              no
 0.34   0.0       0.3446         0.3463     0.34             yes
 0.40   0.0       0.4813         0.4837     0.40             yes
 0.45   0.0       0.6488         0.6518     0.45             yes
 0.10   0.5       0.0724         0.0724     0.10              no
 0.20   0.5       0.1830         0.1824     0.20              no
 0.25   0.5       0.2501         0.2500     0.25              no
 0.30   0.5       0.3254         0.3269     0.30             yes
 0.34   0.5       0.3953         0.3970     0.34             yes
 0.40   

In [ ]:
# NB23-2: the 2×2 payoff table and the "threshold" — fill a publish/withhold game between two small players with the relative revenue of selfish mining and read off the equilibrium (for study)
# Two pools P (share a) and Q (share b) each choose "honest (publish at once)" or "withhold (selfish mining)". The rest (1−a−b) are honest.
# Simplification: if only one pool withholds, its revenue is R_pool(α, γ); if both withhold, neither knows the other's hidden blocks, so each uses R_pool and the effect of the other also withholding is ignored (limitation stated).
def payoff(a, b, P_selfish, Q_selfish, gamma):
    rp = R_pool_closed_form(a, gamma) if P_selfish else a
    rq = R_pool_closed_form(b, gamma) if Q_selfish else b
    return rp, rq

for gamma in [0.0, 0.5]:
    print(f"\nγ = {gamma}: threshold (1−γ)/(3−2γ) = {(1-gamma)/(3-2*gamma):.3f}")
    for a, b in [(0.20, 0.20), (0.30, 0.30), (0.40, 0.20)]:
        print(f"  P={a:.2f}, Q={b:.2f}  (rows: P's choice; columns: Q's choice; cells: P's revenue, Q's revenue)")
        print(f"  {'':>10}{'Q honest':>18}{'Q withhold':>18}")
        for ps, pl in [(False, "P honest"), (True, "P withhold")]:
            cells = []
            for qs in [False, True]:
                rp, rq = payoff(a, b, ps, qs, gamma)
                cells.append(f"({rp:.3f}, {rq:.3f})")
            print(f"  {pl:>10}{cells[0]:>18}{cells[1]:>18}")
        best_P = "withhold" if R_pool_closed_form(a, gamma) > a else "honest"
        best_Q = "withhold" if R_pool_closed_form(b, gamma) > b else "honest"
        print(f"  → P's dominant strategy {best_P}, Q's dominant strategy {best_Q}  (below the threshold share honest is dominant, above it withhold is)")
print("\nThis table rests on the simplification 'whatever the other does, my revenue is set by the threshold'. The interaction when both pools withhold at the same time is outside the model.")


γ = 0.0: threshold (1−γ)/(3−2γ) = 0.333
  P=0.20, Q=0.20  (rows: P's choice; columns: Q's choice; cells: P's revenue, Q's revenue)
                      Q honest        Q withhold
    P honest    (0.200, 0.200)    (0.200, 0.130)
  P withhold    (0.130, 0.200)    (0.130, 0.130)
  → P's dominant strategy honest, Q's dominant strategy honest  (below the threshold share honest is dominant, above it withhold is)
  P=0.30, Q=0.30  (rows: P's choice; columns: Q's choice; cells: P's revenue, Q's revenue)
                      Q honest        Q withhold
    P honest    (0.300, 0.300)    (0.300, 0.273)
  P withhold    (0.273, 0.300)    (0.273, 0.273)
  → P's dominant strategy honest, Q's dominant strategy honest  (below the threshold share honest is dominant, above it withhold is)
  P=0.40, Q=0.20  (rows: P's choice; columns: Q's choice; cells: P's revenue, Q's revenue)
                      Q honest        Q withhold
    P honest    (0.400, 0.200)    (0.400, 0.130)
  P withhold    (0.484, 0.20